Importing libraries

In [10]:
import duckdb
import pandas as pd

from pathlib import Path
from detector.eval.metrics import summarise, threshold_at_fpr
from detector.models.tfidf import fit, score

In [11]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = duckdb.connect().sql(f"select * from '{ROOT / 'data/processed/essays.parquet'}'").df()

#test stays sealed until compare to final 
train, val = df[df.split == "train"], df[df.split == "val"]
df.columns.tolist()

['text_key',
 'text',
 'label',
 'sources',
 'n_words',
 'cluster_id',
 'topic',
 'split']

In [12]:
print(df.groupby("topic")["label"].agg(rows='size', ai_share="mean").round(3))
for t in sorted(df.topic.unique()):
    print(t, [x[:80].replace("\n", " ") for x in df[df.topic == t].sample(2, random_state=0).text])

       rows  ai_share
topic                
0      1989     0.064
1      3830     0.309
2      4019     0.071
3      1502     0.088
4      3853     0.298
5      1718     0.079
6      3332     0.080
0 ['First of all, the idea that studying Venus is a worthy pursuit despite the dange', 'As seen in paragraph 2 it says how venus is the closest planet to earth in terms']
1 ['Do you know what pollution is? Have you ever taken a trip by car? Well, cars emi', 'Hey there fellow cititzens! Today Im gonna tell you why we should limit car usag']
2 ['Do you think that reading the emotional expressions of students in a classroom i', 'The use of this technology in a classroom would be useful because it can help te']
3 ['A reason that people should participate in the seagoing cowboy program is becaus', "Have you ever thought about participating in the Seagoing Cowboys program? It's "]
4 ['Dear Mr. Senator,  As you are most likely aware, the Electoral College is a long', 'The Electoral College  Dear Mr

In [14]:
#length stress test; train on full essays and only score the first N words
vec, clf = fit(train.text, train.label)
thr_full = threshold_at_fpr(val.label, score(vec, clf, val.text), .01)

rows = []
for n in (30, 60, 100, 200, None):
    texts = val.text if n is None else val.text.map(lambda x: " ".join(x.split()[:n]))
    s = score(vec, clf, texts)
    fixed = summarise(val.label, s, thr_full)
    own = summarise(val.label, s, threshold_at_fpr(val.label, s, .01))
    rows.append({
        "words": n or "full",
        "roc_auc": fixed["roc_auc"],
        "pr_auc": fixed["pr_auc"],
        "tpr_fixed_thr": fixed["tpr"],
        "fpr_fixed_thr": fixed["fpr"],
        "tpr_own_thr": own["tpr"]
    })
pd.DataFrame(rows).round(3)

,words,roc_auc,pr_auc,tpr_fixed_thr,fpr_fixed_thr,tpr_own_thr
0,30,0.976,0.933,0.986,0.396,0.783
1,60,0.998,0.991,0.998,0.214,0.963
2,100,1.000,0.999,0.998,0.114,0.998
3,200,1.000,1.000,0.998,0.026,0.998
4,full,1.000,1.000,1.000,0.010,1.000


In [15]:
pool = df[df.split.isin(["train", "val"])]
rows = []

for t in sorted(df.topic.unique()):
    tr = df[(df.split == "train") & (df.topic != t)]
    cal = df[(df.split == "val") & (df.topic != t)]     #threshold comes frm the other topics
    held = pool[pool.topic == t]
    v, c = fit(tr.text, tr.label)
    thr = threshold_at_fpr(cal.label, score(v, c, cal.text), .01)
    r = summarise(held.label, score(v, c, held.text), thr)
    rows.append({"held_out_topic": t, "n": len(held), "ai": int(held.label.sum()), **r})
pd.DataFrame(rows).round(3)

,held_out_topic,n,ai,roc_auc,pr_auc,tpr,fpr
0,0,1699,112,0.999,0.990,1.000,0.113
1,1,3253,1013,0.999,0.998,0.953,0.004
2,2,3403,241,0.985,0.926,0.913,0.033
3,3,1281,107,0.993,0.952,0.308,0.000
4,4,3278,979,0.977,0.950,0.407,0.000
5,5,1455,116,0.999,0.996,0.991,0.000
6,6,2838,219,0.996,0.964,0.950,0.014
